In [ ]:
import pandas as pd
import torch
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Configurações Iniciais

In [ ]:
DATA_DIR = "../data"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


# 2. Carregar os Mapeamentos 

In [ ]:
label_mapping_binario = {'AI': 0, 'Human': 1}
reverse_mapping_ai = {0: 'Anthropic', 1: 'Google', 2: 'Meta', 3: 'OpenAI'} 

# 3. Carregar o Modelo 1 (Binário)

In [ ]:
print("A carregar Modelo Binário...")
modelo_binario = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
modelo_binario.load_state_dict(torch.load('modelo_binario_human_vs_ai.pt'))
modelo_binario.eval()

# 4. Carregar o Modelo 2 (Multiclasse)

In [ ]:
print("A carregar Modelo Multiclasse...")
modelo_multiclasse = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4).to(device)
modelo_multiclasse.load_state_dict(torch.load('modelo_multiclasse_so_ai.pt'))
modelo_multiclasse.eval()

# 5. A Lógica de Previsão em Cascata

In [ ]:
def prever_texto_em_cascata(texto):
    inputs = tokenizer(str(texto), return_tensors="pt", truncation=True, padding='max_length', max_length=200).to(device)
    
    with torch.no_grad():
        # PASSO 1: 
        outputs_bin = modelo_binario(**inputs)
        pred_bin = torch.argmax(outputs_bin.logits, dim=1).item()
        
        # Se for 1, é Humano! Acabou a pesquisa.
        if pred_bin == 1:
            return "Human"
            
        # PASSO 2: 
        outputs_multi = modelo_multiclasse(**inputs)
        pred_multi = torch.argmax(outputs_multi.logits, dim=1).item()
        
        # Mapear o número (0,1,2,3) para o nome da IA
        return reverse_mapping_ai[pred_multi]

# 6. Gerar a Submissão

In [ ]:
ficheiro_teste = "subm3.csv" 
print(f"\nA ler o ficheiro {ficheiro_teste} e a gerar previsões. Isto pode demorar uns minutos...")

df_teste = pd.read_csv(os.path.join(DATA_DIR, ficheiro_teste), sep=';')


# Aplicar a nossa função de cascata a todos os textos!
df_teste['Label'] = df_teste['Text'].apply(prever_texto_em_cascata)
df_submissao = df_teste[['ID','Text','Label']]
print("\nDistribuição Final das Previsões:")
print(df_submissao['Label'].value_counts())

df_submissao = df_teste[['ID', 'Text', 'Label']] 
df_submissao.to_csv("subm3-g4-MIA-A.csv", index=False)

print("\n CONCLUÍDO! O ficheiro 'subm3-g4-MIA-A.csv' foi gerado com sucesso!")